# Silver Layer - Data Transformation

This notebook transforms bronze tables into silver tables with data quality improvements and derived columns.

In [0]:
from pyspark.sql.functions import col, when

## Constants

In [0]:
CATALOG_NAME = 'workspace'
SCHEMA_NAME__BRONZE = 'olist_bronze'
SCHEMA_NAME__SILVER = 'olist_silver'

# Create Silver Schema

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")

qualified_silver_schema_name = f"{CATALOG_NAME}.{SCHEMA_NAME__SILVER}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qualified_silver_schema_name}")

# Create Master Tables

## silver.Status (Pass Through)

In [0]:
# Pass through from bronze - no transformations needed
table__bronze__status = f"{CATALOG_NAME}.{SCHEMA_NAME__BRONZE}.status"
table__silver__status = f"{CATALOG_NAME}.{SCHEMA_NAME__SILVER}.status"

status_df = spark.table(table__bronze__status)

status_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(table__silver__status)

## silver.Products (Add Active Flag)

In [0]:
# Transform products: remove deleted_at, add active boolean column
table__bronze__products = f"{CATALOG_NAME}.{SCHEMA_NAME__BRONZE}.products"
table__silver__products = f"{CATALOG_NAME}.{SCHEMA_NAME__SILVER}.products"

products_bronze_df = spark.table(table__bronze__products)

# Add active column and drop deleted_at
products_silver_df = products_bronze_df \
  .withColumn("active", when(col("deleted_at").isNull(), True).otherwise(False)) \
  .drop("deleted_at")

products_silver_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(table__silver__products)

## silver.Customers (Add Active Flag)

In [0]:
# Transform customers: remove deleted_at, add active boolean column
table__bronze__customers = f"{CATALOG_NAME}.{SCHEMA_NAME__BRONZE}.customers"
table__silver__customers = f"{CATALOG_NAME}.{SCHEMA_NAME__SILVER}.customers"

customers_bronze_df = spark.table(table__bronze__customers)

# Add active column and drop deleted_at
customers_silver_df = customers_bronze_df \
  .withColumn("active", when(col("deleted_at").isNull(), True).otherwise(False)) \
  .drop("deleted_at")

customers_silver_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable(table__silver__customers)

# Verify Silver Tables

In [0]:
%sql
SELECT type, COUNT(*) AS total_status 
FROM workspace.olist_silver.status 
GROUP BY type;

In [0]:
%sql
SELECT 
  active,
  COUNT(*) AS product_count
FROM workspace.olist_silver.products
GROUP BY active;

In [0]:
%sql
SELECT 
  active,
  COUNT(*) AS customer_count
FROM workspace.olist_silver.customers
GROUP BY active;